In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

#Load environment variables 
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew


### Set OpenAI model

In [2]:
os.environ['OPENAI_MODEL_NAME'] = "gpt-4o-mini"

### Load the task and agent ymal file

In [3]:
#Define the file path for yaml configuration
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

#Load configuration from yaml file
configs = {}

for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

#Assign loaded configuratio to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

### Create Pydantic models for structured output

In [5]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of the task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of task with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones") 


### Create Crew, Agents and Tasks

In [ ]:
#Creating Agents
project_planning_agent = Agent(
    config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
    config = agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
    config = agents_config['resource_allocation_agent']
)

#Creating task
task_breakdown = Task(
    config=tasks_config['task_breakdown'],
    agent = project_planning_agent
)

time_resource_estimation = Task(
    config= tasks_config['time_resource_estimation'],
    agent=estimation_agent
)

resource_allocation = Task(
    config=tasks_config['resource_allocation'],
    agent=resource_allocation_agent,
    output_pydantic=ProjectPlan # This is the structured output we want
)

#Creating crew
crew = Crew(
    agents=[
        project_planning_agent,
        estimation_agent, 
        resource_allocation_agent
    ],
    task=[
        task_breakdown,
        time_resource_estimation,
        resource_allocation
    ],
    verbose=True
)

### Crew's Inputs 


In [6]:
from IPython.display import display, Markdown

project = "Website"
industry = 'Technology'
project_objectives = 'Create a webiste for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

#formate the disctionary as markdown for a better display in jupyter lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}

**Project Requirements:**
{project_requirements}

"""

#Display the formatted output as markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a webiste for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)


**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust




### Kicking off the crew

In [7]:
#The given pythong dictionary
inputs = {
    'project_type': project,
    'project_objectives': project_objectives,
    'industry': industry,
    'team_members': team_members,
    'project_requirements': project_requirements
}

#Run the crew
result = crew.kickoff(
    inputs=inputs
)

NameError: name 'crew' is not defined